# Run and track training scripts

In [3]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.ai.ml.entities import AmlCompute

from dotenv import load_dotenv
import os

load_dotenv("../.env", override=True)

subscription_id = os.environ["ARM_SUBSCRIPTION_ID"]
resource_group = os.environ["ARM_RESOURCE_GROUP"]
workspace = os.environ["AZ_ML_NAME"]
mlflow_tracking_uri = os.environ["MLFLOW_TRACKING_URI"]

ml_client = MLClient(
    DefaultAzureCredential(), subscription_id, resource_group, workspace
)

Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


In [4]:
compute_name = "cpu-training-cluster"

try:
    cluster = AmlCompute(
        name=compute_name,
        size="STANDARD_DS3_v2",
        min_instances=0,
        max_instances=1,
        idle_time_before_scale_down=120,   # seconds
        tier="Dedicated",                  # or "LowPriority"
    )

    ml_client.compute.begin_create_or_update(cluster).result()

except Exception as e:
    raise e


In [5]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

file_name = "diabetes-csv"
version = "1"

try:
    csv_asset = Data(
        name=file_name,
        version=version,
        type=AssetTypes.URI_FILE,
        path="../data/train/diabetes.csv", 
        description="Diabetes training data CSV",
    )

    ml_client.data.create_or_update(csv_asset)

except Exception as e:
    if "already exists" in str(e).lower():
        print(f"'{file_name}' with version {version} already exists.")
    else:
        raise e

'diabetes-csv' with version 1 already exists.


In [ ]:
from azure.ai.ml.entities import Environment

env = Environment(
    name="mlflow-env",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu22.04",
    conda_file="../mlops/diabetes/environments/conda.yml",
    description="MLflow training environment",
)

ml_client.environments.create_or_update(env)

Environment({'arm_type': 'environment_version', 'latest_version': None, 'image': 'mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu22.04', 'intellectual_property': None, 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'mlflow-env', 'description': 'MLflow training environment', 'tags': {}, 'properties': {'azureml.labels': 'latest'}, 'print_as_yaml': False, 'id': '/subscriptions/e46b15c2-60e4-419f-a352-ee019a3b0daf/resourceGroups/rg-az-ml/providers/Microsoft.MachineLearningServices/workspaces/mlw-az-ml/environments/mlflow-env/versions/1', 'Resource__source_path': '', 'base_path': '/home/dgouwy/projects/mlops-azure/notebooks', 'creation_context': <azure.ai.ml.entities._system_data.SystemData object at 0x7a1703037290>, 'version': '1', 'conda_file': {'channels': ['conda-forge'], 'dependencies': ['python=3.10', 'pip', {'pip': ['numpy', 'pandas', 'scikit-learn', 'matplotlib', 'mlflow', 'azureml-mlflow']}], 'name': 'mlflow-env'}, 'build': None, 'inferen

In [10]:
from azure.ai.ml import command, Input
from azure.ai.ml.constants import AssetTypes

# configure job
job = command(
    code="../mlops/diabetes/code/",
    inputs={
        "input_data": Input(type=AssetTypes.URI_FILE, path="azureml:diabetes-csv:1")
    },
    command="python train.py --training_data ${{inputs.input_data}}",
    environment="AzureML-sklearn-0.24-ubuntu18.04-py37-cpu@latest",
    compute=compute_name,
    display_name="train-model",
    experiment_name="train-classification-model"
    )

returned_job = ml_client.create_or_update(job)

In [ ]:
# Delete any existing computes:
ml_client.compute.begin_delete(compute_name).result()